# 📰 Fake News Detector — Model Training
### Uses files uploaded directly to Colab (no Google Drive needed)
**Your files:** Fake.csv, True.csv, WELFake_Dataset.csv, gossipcop_fake.csv, gossipcop_real.csv, BuzzFeed files

In [ ]:
# STEP 1 — Install libraries
!pip install scikit-learn joblib nltk matplotlib seaborn --quiet
print('✅ Libraries installed!')

In [ ]:
# STEP 2 — Import everything
import pandas as pd
import numpy as np
import re, os, glob, joblib
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
STOP_WORDS = set(stopwords.words('english'))
print('✅ Imports done!')

In [ ]:
# STEP 3 — Check uploaded files
# Files you upload in Colab go to /content/
print('Files in /content/:')
for f in sorted(os.listdir('/content/')):
    if f.endswith('.csv'):
        size = os.path.getsize(f'/content/{f}') / (1024*1024)
        print(f'  ✅ {f}  ({size:.1f} MB)')

In [ ]:
# STEP 4 — Load all datasets
frames = []

# ── Helper to safely load a CSV ──────────────────────────────────────────────
def load_csv(path):
    for enc in ['utf-8', 'latin-1', 'cp1252']:
        try:
            return pd.read_csv(path, encoding=enc)
        except:
            pass
    return None

# ── 1. ISOT  (Fake.csv + True.csv) ──────────────────────────────────────────
for fname, lbl in [('Fake.csv', 1), ('True.csv', 0)]:
    path = f'/content/{fname}'
    if os.path.exists(path):
        df = load_csv(path)
        if df is not None:
            df['label'] = lbl
            df['text']  = df.get('title', pd.Series()).fillna('') + ' ' + df.get('text', pd.Series()).fillna('')
            df['category'] = 'Politics'
            frames.append(df[['text','label','category']])
            print(f'✅ ISOT {fname}: {len(df):,} rows')
    else:
        print(f'⚠️  {fname} not found')

# ── 2. WELFake ───────────────────────────────────────────────────────────────
path = '/content/WELFake_Dataset.csv'
if os.path.exists(path):
    df = load_csv(path)
    if df is not None:
        print('WELFake columns:', df.columns.tolist())
        # Fix column names if needed
        df.columns = [c.strip().lower() for c in df.columns]
        if 'label' not in df.columns:
            df.rename(columns={df.columns[-1]: 'label'}, inplace=True)
        text_col = 'text' if 'text' in df.columns else ('statement' if 'statement' in df.columns else df.columns[1])
        title_col = 'title' if 'title' in df.columns else None
        df['text'] = (df[title_col].fillna('') + ' ' if title_col else '') + df[text_col].fillna('')
        df['label'] = pd.to_numeric(df['label'], errors='coerce').fillna(0).astype(int)
        df['category'] = 'General'
        frames.append(df[['text','label','category']])
        print(f'✅ WELFake: {len(df):,} rows | labels: {df["label"].value_counts().to_dict()}')
else:
    print('⚠️  WELFake_Dataset.csv not found')

# ── 3. GossipCop (Entertainment) ─────────────────────────────────────────────
for fname, lbl in [('gossipcop_fake.csv', 1), ('gossipcop_real.csv', 0)]:
    path = f'/content/{fname}'
    if os.path.exists(path):
        df = load_csv(path)
        if df is not None:
            df.columns = [c.strip().lower() for c in df.columns]
            print(f'GossipCop {fname} columns:', df.columns.tolist())
            # Find best text column
            for col in ['title','news_url','id','text']:
                if col in df.columns:
                    df['text'] = df[col].fillna('').astype(str)
                    break
            df['label'] = lbl
            df['category'] = 'Entertainment'
            frames.append(df[['text','label','category']])
            print(f'✅ {fname}: {len(df):,} rows')
    else:
        print(f'⚠️  {fname} not found')

# ── 4. BuzzFeed ───────────────────────────────────────────────────────────────
for fname, lbl in [('BuzzFeed_fake_news_content.csv', 1), ('BuzzFeed_real_news_content.csv', 0)]:
    # try both with and without .csv extension
    for try_name in [fname, fname.replace('.csv','')]:
        path = f'/content/{try_name}'
        if os.path.exists(path):
            df = load_csv(path)
            if df is not None:
                df.columns = [c.strip().lower() for c in df.columns]
                print(f'BuzzFeed columns:', df.columns.tolist()[:6])
                for col in ['text','title','body','content']:
                    if col in df.columns:
                        df['text'] = df[col].fillna('').astype(str)
                        break
                df['label'] = lbl
                df['category'] = 'General'
                frames.append(df[['text','label','category']])
                print(f'✅ {try_name}: {len(df):,} rows')
            break

print(f'\n📊 Total frames loaded: {len(frames)}')

In [ ]:
# STEP 5 — Combine + add synthetic samples for missing categories
if not frames:
    raise Exception('❌ No CSV files loaded! Make sure you uploaded them to Colab.')

df_all = pd.concat(frames, ignore_index=True)
print(f'Combined size: {len(df_all):,} rows')
print(df_all['label'].value_counts())
print(df_all['category'].value_counts())

# Synthetic samples — ensures all categories work
synthetic = [
    # Health - Fake
    ('Doctors reveal drinking bleach cures all diseases instantly', 1, 'Health'),
    ('Vaccines contain microchips that track your location for government surveillance', 1, 'Health'),
    ('New miracle cure eliminates cancer in 24 hours secret government hiding it', 1, 'Health'),
    ('5G towers spreading deadly radiation causing mass illness worldwide', 1, 'Health'),
    ('Eating rocks daily boosts immune system by five hundred percent doctors confirm', 1, 'Health'),
    # Health - Real
    ('CDC recommends annual flu vaccines for all individuals aged six months and older', 0, 'Health'),
    ('New clinical trial shows promising results for Alzheimer treatment medication', 0, 'Health'),
    ('WHO publishes updated guidelines on healthy diet and physical activity for adults', 0, 'Health'),
    ('Researchers discover new antibiotic effective against drug resistant bacteria strains', 0, 'Health'),
    ('Harvard study finds regular exercise reduces risk of heart disease significantly', 0, 'Health'),
    # Tech - Fake
    ('Apple secretly installed backdoors in all iPhones to spy on users for CIA', 1, 'Tech'),
    ('Google admits Android phones record all conversations and sell data to advertisers', 1, 'Tech'),
    ('Microsoft Windows update secretly mines bitcoin using your computer resources', 1, 'Tech'),
    ('New smartphone battery explodes if charged more than fifty percent company hiding danger', 1, 'Tech'),
    # Tech - Real
    ('Apple announced its latest processor achieves significant performance improvements over previous generation', 0, 'Tech'),
    ('Google releases updated privacy dashboard giving users more control over their data', 0, 'Tech'),
    ('OpenAI published research paper on improving large language model reasoning capabilities', 0, 'Tech'),
    ('Microsoft Azure cloud services reported strong quarterly growth exceeding analyst expectations', 0, 'Tech'),
    # History - Fake
    ('Newly discovered documents prove Napoleon was actually six feet tall and a woman', 1, 'History'),
    ('Secret files reveal World War Two was staged by Hollywood directors for propaganda purposes', 1, 'History'),
    ('Ancient scrolls prove Egypt pyramids were built by aliens archaeologist was silenced', 1, 'History'),
    # History - Real
    ('Archaeologists discovered a well preserved Roman villa beneath a Spanish vineyard this year', 0, 'History'),
    ('New analysis of ancient DNA reveals migration patterns of early human populations across continents', 0, 'History'),
    ('Historians published revised account of Silk Road trading routes economic and cultural impact', 0, 'History'),
    # Sports - Fake
    ('FIFA confirms World Cup results were fixed by secret international betting syndicate for years', 1, 'Sports'),
    ('Star athlete secretly uses experimental drug undetected by all current anti doping tests', 1, 'Sports'),
    # Sports - Real
    ('The national football team secured a two one victory in the international qualifier match', 0, 'Sports'),
    ('Olympic committee announced new stricter anti doping regulations for upcoming summer games', 0, 'Sports'),
    ('Premier League club breaks transfer record signing midfielder for record fee this summer', 0, 'Sports'),
]

# Repeat 60x to give good weight
syn_df = pd.DataFrame(synthetic * 60, columns=['text','label','category'])
df_all = pd.concat([df_all, syn_df], ignore_index=True)
print(f'\nAfter synthetic samples: {len(df_all):,} rows')
print(df_all['category'].value_counts())

In [ ]:
# STEP 6 — Preprocess text
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [w for w in text.split() if w not in STOP_WORDS and len(w) > 2]
    return ' '.join(tokens)

print('Preprocessing text... (this may take 1-2 minutes)')
df_all['clean_text'] = df_all['text'].apply(preprocess)
df_all = df_all[df_all['clean_text'].str.len() > 10]
df_all = df_all.dropna(subset=['clean_text','label'])
print(f'✅ After cleaning: {len(df_all):,} rows')

In [ ]:
# STEP 7 — Train / Test Split
X = df_all['clean_text']
y = df_all['label'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

In [ ]:
# STEP 8 — Train Model
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=2
    )),
    ('clf', LogisticRegression(
        max_iter=1000, C=5.0, solver='lbfgs', n_jobs=-1
    ))
])

print('Training model...')
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f'\n🎯 Accuracy: {accuracy * 100:.2f}%')
print(classification_report(y_test, y_pred, target_names=['Real','Fake']))

if accuracy >= 0.80:
    print('✅ 80%+ accuracy ACHIEVED!')
else:
    print('⚠️ Below 80% — but model is still saved. Try with more data.')

In [ ]:
# STEP 9 — Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Real','Fake'], yticklabels=['Real','Fake'])
plt.title(f'Confusion Matrix — Accuracy: {accuracy*100:.2f}%')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# STEP 10 — Save & Download Model
joblib.dump(pipeline, '/content/fake_news_model.pkl')
print('Model saved!')

from google.colab import files
files.download('/content/fake_news_model.pkl')
files.download('/content/confusion_matrix.png')
print('\n✅ Download started!')
print('👉 Place fake_news_model.pkl inside the backend/ folder')

In [ ]:
# STEP 11 — Quick Test
tests = [
    ('Drinking bleach cures COVID according to secret government documents', 'Health'),
    ('CDC recommends annual flu vaccine for all adults and children', 'Health'),
    ('Apple secretly records users through disabled microphones CIA confirmed', 'Tech'),
    ('Google announces new AI model achieving state of the art benchmark results', 'Tech'),
    ('Hollywood actor replaced by clone ten years ago insiders confirm shocking truth', 'Entertainment'),
    ('Netflix quarterly earnings exceed analyst expectations subscriber growth strong', 'Entertainment'),
    ('FIFA World Cup results fixed by international betting syndicate for years', 'Sports'),
    ('Olympic committee announces new anti doping rules for upcoming games', 'Sports'),
]
print('--- Quick Test ---')
for text, cat in tests:
    clean = preprocess(text)
    pred = pipeline.predict([clean])[0]
    prob = pipeline.predict_proba([clean])[0]
    print(f'[{cat}] {"FAKE" if pred==1 else "REAL"} ({max(prob)*100:.1f}%) → {text[:55]}...')